# Metabolic model consisteny quick-check (mass and charge balance)

This notebook provides a fast way to check genome-scale metabolic model (GSMM) consistency after you make small edits in correcting mass and charge balance.

Use this to check the imbalances in a model while fixing mass and charge balance, instead of running MEMOTE report every time. MEMOTE is however recommended if you need a full model quality report.

This provides feedback on:
1. Mass imbalance (elemental balance such as C, H, N, O, P, S, and R.)
2. Charge imbalance (when net charge does not balance)

Workflow
1. Load the model
2. Run FBA - maximize biomass (Sanity check that the model grows)
3. Run mass and charge balance checks across all reactions
4. Export the imbalanced reactions to a CSV for review in Excel
5. Inspect any individual reaction in detail

In [ ]:
from pathlib import Path
# Path relative to the repository (notebook lives in code/1_curation/).
MODEL_PATH = Path.cwd().parents[1] / "models" / "02_curated_iMMU1867" / "iMMU1867.xml"

# Imports

import cobra
import pandas as pd
from cobra.manipulation.validate import (check_mass_balance)

In [10]:
# Load model

model = cobra.io.read_sbml_model(str(MODEL_PATH))

In [11]:
# Summary of model composition

print(f"Model: {model.id}")
print(f"Reactions: {len(model.reactions)}")
print(f"Metabolites: {len(model.metabolites)}")
print(f"Genes: {len(model.genes)}")

Model: iMM1865
Reactions: 10612
Metabolites: 5839
Genes: 1865


In [ ]:
# Run FBA to check if the model is functional

BIOMASS_REACTION = "BIOMASS_reaction"  # Update with your model's biomass reaction ID
model.objective = BIOMASS_REACTION

solution = model.optimize()
print("Objective value:", solution.objective_value)

Objective value: 798.8110517749973


In [13]:
# Check mass/charge balance of all reactions
# the imbalance dictionary contains keys like 'C', 'H', 'O', 'N', 'charge' etc.

mass_balance_issues = check_mass_balance(model)
print(f"Number of reactions with any imbalance: {len(mass_balance_issues)}\n")

# Show first 10 imbalanced reactions
for i, (rxn, imbalance) in enumerate(mass_balance_issues.items()):
    print(rxn.id, imbalance)
    if i >= 9:
        break


Number of reactions with any imbalance: 4895

1a_24_25VITD2Hm {'charge': 1.0}
1a_25VITD2Hm {'charge': 1.0}
1MNCAMti {'charge': 1.0}
1PPDCRp {'charge': 1.0}
24_25VITD2Hm {'charge': 1.0}
24_25VITD3Hm {'charge': 1.0}
25VITD2Hm {'charge': 1.0}
25VITD3Hm {'charge': 1.0}
2AMACSULT {'charge': 4.0, 'O': 3.0, 'S': 1.0}
2DR1PP {'charge': 2.0}


In [14]:
# Export all imbalanced reactions to CSV for Excel
# You can rerun this cell after each small batch of fixes.
# Comment out if not needed.

# Collect all imbalance keys (C, H, N, O, P, R, S, X, Charge)
all_keys = set()
for imb in mass_balance_issues.values():
    all_keys.update(imb.keys())
all_keys = sorted(all_keys)

rows = []
for rxn, imb in mass_balance_issues.items():
    row = {
        "reaction_id": rxn.id,
        "reaction_name": rxn.name,
        "equation": rxn.reaction,
    }
    # Fill each key with its imbalance value, default 0
    for key in all_keys:
        row[key] = imb.get(key, 0.0)
    rows.append(row)

df_imbalance = pd.DataFrame(rows)

# Save to CSV
output_csv = "mass_charge_imbalances.csv"
df_imbalance.to_csv(output_csv, index=False)

print(f"Saved imbalance table to {output_csv}")
df_imbalance.head()

Saved imbalance table to mass_charge_imbalances.csv


,reaction_id,reaction_name,equation,Ba,C,Ca,Cl,Co,F,Fe,...,N,Na,O,P,R,S,X,Y,Zn,charge
0,1a_24_25VITD2Hm,"1-alpha-Vitamin D-24,25-hydroxylase (D2)",2425dhvitd2_m + h_m + nadph_m + o2_m --> 1a242...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,1a_25VITD2Hm,"1-alpha,24R,25-Vitamin D-hydroxylase (D2)",1a25dhvitd2_m + h_m + nadph_m + o2_m --> 1a242...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1MNCAMti,N1-Methylnicotinamide transport,1mncam_c + atp_c + h2o_c --> 1mncam_e + adp_c ...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1PPDCRp,"Delta1-piperideine-2-carboxylate reductase, pe...",1pipdn2c_x + h_x + nadh_x --> Lpipecol_x + nad_x,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,24_25VITD2Hm,24R-Vitamin D-25-hydroxylase (D2),25hvitd2_m + h_m + nadph_m + o2_m --> 2425dhvi...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [15]:
# Separate charge-only issues from mass-balance issues

charge_imb_rxn = {}
mass_imb_rxn = {}

for rxn, imbalance in mass_balance_issues.items():
    keys = set(imbalance.keys())
    if keys == {"charge"}:
        charge_imb_rxn[rxn.id] = imbalance["charge"]
    else:
        mass_imb_rxn[rxn.id] = imbalance

print(f"Total imbalanced reactions: {len(mass_balance_issues)}")
print(f"Charge-only issues: {len(charge_imb_rxn)}")
print(f"Mass-balance issues: {len(mass_imb_rxn)}")


Total imbalanced reactions: 4895
Charge-only issues: 2791
Mass-balance issues: 2104


In [ ]:
# Function to inspect a reaction in detail

def inspect_reaction(model, rxn_id):
    rxn = model.reactions.get_by_id(rxn_id)
    print(f"Reaction ID: {rxn.id}")
    print(f"Name: {rxn.name}")
    print(f"Equation: {rxn.reaction}")
    print("\nMetabolites (coeff, id, formula, charge):")
    for met, coeff in rxn.metabolites.items():
        print(f"{coeff:>6}  {met.id:15}  {met.formula:15}  charge={met.charge}")

In [ ]:
# Inspect one imbalanced reaction 

rxn_ID = "R08180"  # change rxn_ID to any imbalanced reaction
inspect_reaction(model, rxn_ID)

print("\nMass imbalance:", mass_imb_rxn.get(rxn_ID))
print("\nCharge imbalance:", charge_imb_rxn.get(rxn_ID))

In [ ]:
# Summary of consistency status

print("--- Summary ---")
print(f"Imbalanced reactions in model: {len(mass_balance_issues)}")
print(f"Charge-only issues: {len(charge_imb_rxn)}")
print(f"Mass-balance issues: {len(mass_imb_rxn)}")
